# Разбор A. Электронная модель

Механическое поле не описывает перераспределение электронов. Прежде чем читать квантовый результат истрадефиллина, воспроизведём ограниченный RHF-контроль H₂. Гипотеза: энергия зависит от межъядерного расстояния, а код должен одновременно контролировать плотность и энергию. H₂ проверяет метод, но не сообщает свойства A2A-лиганда или аторвастатина.

Запустите существующее прозрачное ядро h2_scf на 0.74, 1.0 и 1.5 Å. Сохраните трассы итераций, проверьте известный контроль около −1.11676 Eh на 0.74 Å. Затем попытайтесь ограничить число итераций единицей и зафиксируйте отказ. Обсудите невозможность переноса энергий H₂ на лекарственные объекты.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M07-L01','case')


Это предварительный контроль RHF/STO-3G на H₂, а не завершённое исследование лекарственного объекта. До запуска различите три расстояния H–H, энергию в Eh и признак сходимости. Проверяем, способен ли вычислительный алгоритм пройти известный малый контроль; перенос на истрадефиллин или B из него не следует.

Перед большим квантовым расчётом я хочу проверить маленькую систему, в которой можно проследить алгоритм и намеренно вызвать отказ. SCF — согласование орбиталей и плотности с оператором, который сам зависит от плотности. Последнее напечатанное значение энергии ещё не сообщает, выполнен ли критерий этой согласованности.

Поэтому в контроле H2 я смотрю одновременно на расстояние, энергию, число итераций и объявленную сходимость. Ограничение одной итерацией — проверка поведения при недостаточном решении. Успех проверки заключается в обнаруженном отказе, а не в попытке использовать незавершённую энергию как точный результат.

In [3]:
from h2_scf import rhf_h2
rows=[rhf_h2(r) for r in [.74,1.,1.5]]
assert all(r['converged'] for r in rows)
assert abs(rows[0]['energy_hartree']-(-1.1167593))<1e-5
display([{k:r[k] for k in ['distance_angstrom','energy_hartree','cycles']} for r in rows])


[{'distance_angstrom': 0.74,
  'energy_hartree': -1.1167593073781579,
  'cycles': 2},
 {'distance_angstrom': 1.0,
  'energy_hartree': -1.0661086491853147,
  'cycles': 2},
 {'distance_angstrom': 1.5,
  'energy_hartree': -0.9108735543822694,
  'cycles': 2}]

Ограничим SCF одной итерацией. Ожидаемый результат — отказ, потому что последнее напечатанное значение энергии ещё не означает сходимость. Затем прочитайте кривую как зависимость энергии выбранной модели от расстояния, а не как свободную энергию реакции или подтверждение корректного описания диссоциации.

In [4]:
try:rhf_h2(.74,max_cycles=1)
except RuntimeError as error:print('Expected nonconvergence:',error)
else:raise AssertionError('One cycle accepted')
fig,ax=plt.subplots();ax.plot([r['distance_angstrom'] for r in rows],[r['energy_hartree'] for r in rows],'o-')
ax.set(xlabel='H-H / Angstrom',ylabel='RHF/STO-3G energy / Eh');publish_plot(fig,'h2.png')


Expected nonconvergence: SCF did not converge


<Figure size 640x480 with 1 Axes>

Сохраняем именно статус контрольной задачи H₂. Профессиональная часть требует отдельно завершённых расчётов целой целевой формы, численных проверок и анализа геометрии; этот контроль не закрывает её отсутствие.

In [5]:
result={'control':'H2 only','runs':rows,'kind':'calculated','drug_QM':'not inferred from H2'}


Теперь я могу отдельно назвать две причины неудовлетворительного результата: уравнения выбранной модели ещё не решены с нужной точностью либо сама модель недостаточна для физической задачи. Контроль SCF работает с первой причиной. Выбор метода и базиса для целого лекарства, геометрия, среда и проверка состояния требуют следующих шагов. Я переношу в B порядок контроля расчёта, сохраняя границы малого примера.

In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M07-L01','A',result,ROOT,inputs=[])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\reasoning-full-20261120\workspace\my_research\evidence\M07-L01\20261120T105347458002Z-A-924de4a2074a4e0f9f92a2c61fbfe6ae.json


Кривая показывает зависимость энергии в выбранном базисе, не свободную энергию реакции. Для реального B нужно отдельно задать заряд, spin=2S, геометрию и метод. Следующий урок проверит именно паспорт полного расчёта.

Что вы сообщите коллеге, если энергия напечатана, но сходимость не достигнута? Какие проверки ещё нужны для расчёта B, даже если контроль H₂ прошёл?